# Context Engineering

**What you will build:** the three mechanisms that keep a long agent session alive — **compaction** (summarize the past in place), a **subagent** tool (burn tokens in someone else's window), and a **todo list** (a plan that survives the noise) — each in a few dozen lines on the loop you already own. You will also separate the application's **transcript** from the messages sent to the model, then connect the economics through prompt caching and model routing.

Your coding agent from 0.7 works. This notebook is about why it *stops* working around turn thirty of a real session, and what every production harness does about it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/08_context_engineering.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run once.
!pip install -q openai

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"   # change to any model on openrouter.ai/models
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready. Model:", MODEL)

## Why long sessions die

Two facts collide in every agent:

1. **The message list only grows.** Every turn appends the model's reply plus every tool result — and tool results dominate. One `read_file` of a big module, one verbose pytest traceback, one chatty API response: hundreds or thousands of tokens each, resent on *every* subsequent call (0.1's statelessness, presented with the bill).
2. **A fuller window is a worse window.** Long before you hit the hard limit, quality degrades — models attend less reliably to material buried in a huge context. Anthropic's engineers call the phenomenon **context rot** and describe attention as a *budget* you spend, not a warehouse you fill.

```
turn:        1          5          15                    30
window:  [sys|task] [sys|task|■■■] [sys|task|■■■■■■■■■] [sys|task|■■■■■■■■■■■■■■■■■■■■■■■]
                              ▲ tool results pile up      ▲ the plan from turn 1 is now 2%
                                                            of the window — and the model
                                                            behaves accordingly
```

The scale involved is worth knowing: before you type a single word to Claude Code, its harness has already spent ~2,800 tokens of system prompt plus ~9,400 tokens of tool descriptions. The [MinusX teardown](https://minusx.ai/blog/decoding-claude-code/) measured all of this from intercepted logs; a trivial task cost $0.11 and 40 seconds of overhead. Production agents are context-management machines that occasionally call a model.

Every real harness fights the growth with the same four moves — **compact, delegate, anchor, offload**. We build the first three and discuss the fourth.

## Mechanism 1 — Compaction

When the window gets heavy, summarize the old part *in place*: keep the system prompt, keep the last few messages verbatim, and replace everything in between with a dense summary. This is exactly what Claude Code's auto-compact does when it approaches the limit (you've seen it if you've used it: "Compacting conversation…").

The summary prompt is where compaction lives or dies. It must preserve **decisions, facts, and current state** — not narrative. "The user asked about databases and the assistant explained helpfully" is a useless summary; "user = Alex; project Atlas, deadline Friday; chose Postgres over MySQL; migration script half-written" is a working one.

In [ ]:
SUMMARIZER_MODEL = MODEL   # in production, point this at a small cheap model —
                           # more than half of Claude Code's LLM calls go to one (see routing, below)

def compact(messages, keep_last=4):
    """Replace everything between the system prompt and the last keep_last
    messages with a dense summary the conversation can continue from."""
    if len(messages) <= 1 + keep_last:
        return messages
    head, tail = messages[1:-keep_last], messages[-keep_last:]
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in head)

    summary = client.chat.completions.create(
        model=SUMMARIZER_MODEL, temperature=0,
        messages=[
            {"role": "system", "content":
             "Summarize this conversation so an assistant can CONTINUE it without losing important context. "
             "Preserve: every fact about the user, every decision made, the current state "
             "of any task, and what remains to be done. Dense and factual. No narrative."},
            {"role": "user", "content": transcript},
        ]).choices[0].message.content

    return [messages[0],
            {"role": "user", "content": f"[Conversation so far, compacted]\n{summary}"},
            {"role": "assistant", "content": "Understood — continuing from that state."},
            ] + tail

Now a chat that compacts itself. The threshold is absurdly low (real harnesses compact near the *model's* limit, tens of thousands of tokens) so you can watch it fire within a handful of turns:

In [ ]:
COMPACT_AT = 700    # prompt tokens — tiny on purpose, so compaction fires during the demo

class CompactingChat:
    def __init__(self, system):
        self.messages = [{"role": "system", "content": system}]

    def say(self, text):
        self.messages.append({"role": "user", "content": text})
        r = client.chat.completions.create(model=MODEL, messages=self.messages, temperature=0)
        ptoks = r.usage.prompt_tokens
        reply = r.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        if ptoks > COMPACT_AT:
            before = len(self.messages)
            self.messages = compact(self.messages)
            print(f"  [auto-compact: {before} -> {len(self.messages)} messages, was {ptoks} prompt tokens]")
        print(f"  ({ptoks} prompt tokens, {len(self.messages)} messages in list)")
        return reply

chat = CompactingChat("You are a helpful, thorough assistant.")

# Turn 1 establishes facts that must SURVIVE the compaction:
chat.say("I'm Alex. My project is called Atlas and its deadline is Friday.")

# Verbose middle turns inflate the window (each answer is a few hundred tokens):
chat.say("Explain in about 120 words why Postgres uses MVCC.")
chat.say("Explain in about 120 words what a B-tree index is.")
chat.say("Explain in about 120 words how connection pooling works.")

# After compaction fired: does it still know the turn-1 facts?
print("\n" + chat.say("Quick check — what's my name, my project, and its deadline?"))

The final answer names Alex, Atlas and Friday — facts that no longer exist *verbatim* anywhere in the message list. They survived inside the summary. That's compaction working: the window stays bounded while the conversation's *meaning* continues.

And it's lossy, of course — 0.6's lesson didn't stop being true. Whatever the summarizer didn't consider worth keeping is gone without a trace. This is why Claude Code lets users steer compaction (`/compact focus on the API design`) and why its harness compacts **rarely, in big jumps**, not every turn. There's also a second reason for that timing, and it's about money:

::::{dropdown} 🔧 Under the hood: prompt caching — the numbers promised in 0.6
:color: info

Resending a growing history every turn sounds ruinous. It isn't, because providers cache prompt **prefixes**: if the first N tokens of your request are byte-identical to a recent request, the provider skips recomputing them and bills them at a deep discount — cached input tokens cost **~10× less** on Anthropic's API (OpenAI applies an automatic ~50% discount; details vary, the principle doesn't).

The agent loop is almost perfectly cache-shaped: system prompt and tool schemas never change, and the message list is append-only within a session. Turn 30 re-reads 29 turns of history from cache for pennies. This is what makes "just resend everything" a viable architecture at all.

Now see the tension with everything else in this notebook:

- **Compaction rewrites the front of the list** → full cache bust, next call recomputes everything. Compact *rarely and in big jumps* and you amortize one bust across many cheap cached turns; compact every turn and you pay full price every time (the mistake 0.6's `SummaryMemory` makes, folding on every turn — fine for teaching, wrong for production).
- **A window that slides** (0.6's `WindowMemory`) shifts the prefix every turn → also cache-hostile.
- **Subagents** (next section) are cache-*friendly*: the main conversation stays append-only; the token burn happens in a disposable side context.

One number to retain: cached-read pricing means an agent that *appears* to resend 50,000 tokens per turn is often paying for a small fraction of that — as long as its harness respects the prefix. "Keep the prefix stable" is a first-class design rule in every serious harness.
::::

## Transcript ≠ provider feed

So far, `messages` has done two jobs: it is the application's record of what happened **and** the payload sent to the model. A real harness soon needs those views to diverge. The UI may need timestamps, connector attribution, audit IDs, hidden-result counts, or display-only reasoning. Sending those sidecars back to the model wastes context and can leak information the tool deliberately withheld.

Use one explicit boundary: persist the rich transcript, then derive the provider feed immediately before each model call. The visible tool result stays in `content`; display metadata stays beside it:

In [ ]:
DISPLAY_ONLY = {"ts", "source", "_display", "reasoning", "audit_id"}

def provider_messages(transcript: list[dict]) -> list[dict]:
    """Return fresh model-facing messages without application sidecars."""
    return [{k: v for k, v in message.items() if k not in DISPLAY_ONLY}
            for message in transcript]

transcript = [
    {"role": "user", "content": "Show my urgent emails",
     "ts": "2026-07-24T09:30:00Z", "source": {"surface": "desktop"}},
    {"role": "tool", "tool_call_id": "call_1",
     "content": json.dumps({"emails": ["Launch review at 10:00"]}),
     "_display": {"hidden_by_policy": 7}, "audit_id": "audit_42"},
]

outbound = provider_messages(transcript)
print("PERSISTED:\n", json.dumps(transcript, indent=2))
print("\nSENT TO MODEL:\n", json.dumps(outbound, indent=2))

assert "_display" in transcript[1]
assert "_display" not in outbound[1]
assert "hidden_by_policy" not in json.dumps(outbound)

The transcript still supports the UI and audit trail. The model sees only the one email the policy allowed it to see — not even the count of hidden emails. Keep `provider_messages()` as the **single exit** from stored history to a provider adapter; scattered cleanup rules eventually leak.

This is different from compaction. Compaction decides which *model-relevant history* survives a token budget. Transcript filtering decides which fields were *never model context in the first place*. In 0.9, this function becomes a boundary inside a small harness.

## Mechanism 2 — Subagents: burn tokens in someone else's window

Compaction shrinks history *after* it happened. Subagents prevent it from happening at all: delegate a context-heavy job (read this log, search these files, research this question) to a **fresh agent with its own empty window**, and take back only its final answer. The thousands of tokens of raw material die with the subagent; your conversation grows by a sentence.

This is Claude Code's `Task` tool, and OpenCode's subagents. We can build it with what we have — an agent, after all, is just a function, so an agent can be another agent's *tool*:

```
   MAIN agent window                    SUBAGENT window (disposable)
   [sys | task | ...]                   [sys | "find X in the log"]
        │                                    │ read_file -> 700 tokens
        │  task("find X in the log")         │ read_file -> 700 tokens
        ├───────────────────────────▶        │ reasons over all of it
        │                                    ▼
        ◀── "X was decided in meeting 12" ── answer (30 tokens)
        │
   grows by ~30 tokens                  window discarded
```

First, something worth delegating — a long, boring meeting log with three real facts buried in it:

In [ ]:
from pathlib import Path

WORKSPACE = Path("workspace08").resolve()
WORKSPACE.mkdir(exist_ok=True)

filler = ("routine updates: sprint velocity stable, standups on schedule, no blockers reported, "
          "CI green, code review queue under control, on-call week uneventful.")
lines = [f"Meeting {i:02d} — {filler}" for i in range(1, 41)]
lines[11] = "Meeting 12 — DECISION: migrate the main database from MySQL to Postgres in Q3. Dana leads."
lines[23] = "Meeting 24 — the mobile app redesign is postponed until the Postgres migration completes."
lines[33] = "Meeting 34 — hiring: two backend engineer positions approved to support the migration."
(WORKSPACE / "meeting_log.txt").write_text("\n".join(lines))

print(f"meeting_log.txt: {len(lines)} lines, ~{len((WORKSPACE / 'meeting_log.txt').read_text()) // 4} tokens")

Next, a generic mini-agent — the 0.3 loop as a reusable function that reports what it spent — and a bounded `read_file` for the subagent (0.7's, trimmed):

In [ ]:
def read_file(path: str, offset: int = 1, limit: int = 30) -> str:
    """Read a file with line numbers, `limit` lines starting at `offset`."""
    p = WORKSPACE / path
    if not p.is_file():
        return f"Error: {path} does not exist"
    all_lines = p.read_text().splitlines()
    chunk = all_lines[offset - 1 : offset - 1 + limit]
    out = "\n".join(f"{n:3d}| {l}" for n, l in enumerate(chunk, start=offset))
    remaining = len(all_lines) - (offset - 1 + len(chunk))
    if remaining > 0:
        out += f"\n... ({remaining} more lines — call again with offset={offset + len(chunk)})"
    return out

doc_tools = [{"type": "function", "function": {
    "name": "read_file",
    "description": "Read a file with line numbers. Large files need several calls (use offset).",
    "parameters": {"type": "object", "properties": {
        "path":   {"type": "string"},
        "offset": {"type": "integer"},
        "limit":  {"type": "integer"},
    }, "required": ["path"]}}}]
doc_tool_functions = {"read_file": read_file}


def mini_agent(task_text, tools=None, tool_functions=None,
               system="You are a helpful assistant. Use tools when they help.",
               max_turns=8, label=""):
    """The 0.3 loop as a reusable function. Returns (answer, tokens_spent)."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": task_text}]
    total = 0
    for _ in range(max_turns):
        kwargs = {"model": MODEL, "messages": messages, "temperature": 0}
        if tools:
            kwargs["tools"] = tools
        resp = client.chat.completions.create(**kwargs)
        total += resp.usage.total_tokens
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content, total
        for tc in msg.tool_calls:
            args = tc.function.arguments
            try:
                args = json.loads(args)
                result = tool_functions[tc.function.name](**args)
            except Exception as e:
                result = f"Error: {e}"
            print(f"  {label}[tool] {tc.function.name}({str(args)[:50]}) -> {len(str(result))} chars")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "Stopped: reached max turns.", total

And now the move itself: `task()` wraps a whole subagent run inside an ordinary tool. The main agent gets *only* this tool — it cannot read files itself, which is the point:

In [ ]:
SUB_SPEND = {"tokens": 0}

def task(prompt: str) -> str:
    """Delegate a research question to a subagent with file access.
    Only its short final answer comes back to your context."""
    print(f"    [subagent starts] {prompt!r}")
    answer, tokens = mini_agent(
        prompt, doc_tools, doc_tool_functions,
        system=("You are a research subagent. Use the tools to find the answer in the files. "
                "The workspace contains meeting_log.txt. "
                "Reply with a SHORT final answer, under 60 words."),
        label="    sub ")
    SUB_SPEND["tokens"] += tokens
    print(f"    [subagent done: spent {tokens} tokens in its own window]")
    return answer

main_tools = [{"type": "function", "function": {
    "name": "task",
    "description": ("Delegate a self-contained research question to a subagent that can read "
                    "the project files. Returns its findings as a short answer. Use this instead "
                    "of asking the user for file contents."),
    "parameters": {"type": "object", "properties": {
        "prompt": {"type": "string", "description": "The complete, self-contained question"},
    }, "required": ["prompt"]}}}]

Run it. The main agent gets a question whose answer lives in the log it cannot read:

In [ ]:
answer, main_tokens = mini_agent(
    "What did the team decide about the database migration, and what other work depends on it?",
    main_tools, {"task": task},
    system="You are a project assistant. Delegate anything that requires reading documents to the task tool.")

print("\n" + answer)
print(f"\nmain conversation: {main_tokens} tokens | subagent windows: {SUB_SPEND['tokens']} tokens")

Read the two numbers at the bottom. The subagent chewed through the whole log — several `read_file` calls, each hundreds of tokens — **in its own window**, which no longer exists. The main conversation paid a fraction of that and received two sentences that answer the question. The asymmetry *is* the mechanism: total spend barely changed, but the expensive part landed in a context you never have to carry forward, resend, or compact.

Two production notes. First, the subagent's system prompt demands a **short** final answer — a subagent that returns its whole transcript defeats the purpose (you'll prove this to yourself in the exercises). Second, Claude Code deliberately allows only **one level** of delegation: subagents cannot spawn subagents. The MinusX teardown summarizes the reasoning as *debuggability over multi-agent mishmash* — every extra level of delegation makes failures exponentially harder to trace. Bounded structure beats clever hierarchy; keep that instinct for 2.6, where multi-agent design becomes the whole topic.

## Mechanism 3 — A todo list the model maintains

The subtlest failure of long sessions isn't forgetting facts — it's forgetting the *plan*. The task statement from turn 1 drifts toward 2% of the window (context rot again), and around turn 20 the agent quietly stops working on what you asked and starts riffing on the last tool output. Claude Code's answer is `TodoWrite`: a tool whose only job is to let the model **re-state its own plan**, which re-anchors the plan in the freshest, most-attended part of the context.

It's barely code — deliberately so. The value is in the *ritual*, enforced by the system prompt:

In [ ]:
TODOS = []

def write_todos(todos: list) -> str:
    """Replace the plan. Mark finished items by prefixing them with 'DONE: '."""
    global TODOS
    TODOS = todos
    print("    [plan] " + " | ".join(todos))
    return "Current plan:\n" + "\n".join(f"- {t}" for t in todos)


def create_file(path: str, content: str) -> str:
    """Create or overwrite a small text file in the workspace."""
    (WORKSPACE / path).write_text(content)
    return f"Wrote {path} ({len(content)} chars)"

todo_tools = [
    {"type": "function", "function": {
        "name": "write_todos",
        "description": ("Write or update your step-by-step plan. Call it FIRST on any multi-step task, "
                        "and again after finishing each step, marking finished steps with 'DONE: '."),
        "parameters": {"type": "object", "properties": {
            "todos": {"type": "array", "items": {"type": "string"}},
        }, "required": ["todos"]}}},
    {"type": "function", "function": {
        "name": "create_file",
        "description": "Create or overwrite a text file in the workspace.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}, "content": {"type": "string"},
        }, "required": ["path", "content"]}}},
]

A three-file task, with the ritual mandated by the system prompt:

In [ ]:
answer, spent = mini_agent(
    "Create three files: haiku.txt (a haiku about databases), tagline.txt (a six-word tagline "
    "for a bakery), and README.md (listing both files with a one-line description of each).",
    todo_tools, {"write_todos": write_todos, "create_file": create_file},
    system=("You are a careful assistant. For any task with more than two steps: first call "
            "write_todos with your plan, then execute it step by step, updating the todo list "
            "after each finished step."),
    max_turns=12)

print("\n" + str(answer))
print("\nfinal plan state:", TODOS)

Watch the `[plan]` lines: the model wrote its plan before touching a file, then re-wrote it as steps completed. Nothing about the tool *forces* correct behaviour — `write_todos` just stores a list — but the ritual changes the model's behaviour measurably on long tasks: the plan is always recent, always attended to, and always visible to *you*, which makes an agent that's about to go off the rails easy to catch (and redirect) early.

**The fourth mechanism — offloading — you already saw without naming it.** Those files the agent just wrote are *external memory*: state that lives outside the window, readable on demand, surviving any compaction. Real harnesses lean on this hard — scratch files, `NOTES.md`, research dumps written to disk and re-read selectively. LangChain's "Deep Agents" curriculum teaches exactly this four-piece kit — todos, file offload, subagents, summarization — as the anatomy of agents that handle long tasks. You've now built all four with your own hands.

## Model routing: not every call deserves the big model

Look back at where LLM calls happened in this notebook: the *conversation* needs your best model, but the compaction summarizer? Classifying whether a bash command is safe? Extracting facts for a memory file? Those are cheap, mechanical calls — and production harnesses route them to small models. Measured on Claude Code: **more than half of all its LLM calls go to Haiku** (the small tier), at a fraction of the cost per token.

You wired the seam for this already — `SUMMARIZER_MODEL` at the top of the compaction section. Point it at any cheap model on OpenRouter and compaction gets an order of magnitude cheaper without touching the main conversation's quality. The general rule: **route by job, not by habit** — reasoning and generation to the frontier model, summarizing/classifying/formatting to the small one.

## The map: who does what

| Mechanism | Claude Code | Codex CLI | Aider | LangChain "Deep Agents" |
|---|---|---|---|---|
| **Compaction** | auto-compact near the limit, user-steerable | history compaction | avoided up front: repo-map keeps context small by construction | summarization node |
| **Subagents** | `Task` tool, one level max | — | — | subagent nodes with isolated context |
| **Todo anchor** | `TodoWrite` | plan tool | — | `write_todos` |
| **Offload to files** | notes, memory files | `AGENTS.md` | git history as the record | files as working memory |
| **Routing** | >50% of calls to a small model | — | separate "weak model" setting for commit messages etc. | per-node model choice |

Different products, same four moves plus routing — which is the strongest evidence that these aren't implementation quirks but the actual shape of the problem. For the primary sources, in reading order: [Anthropic on context engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents) (the conceptual frame), [the MinusX teardown](https://minusx.ai/blog/decoding-claude-code/) (the measured reality), and [12-Factor Agents](https://github.com/humanlayer/12-factor-agents) — whose factor 3, *own your context window*, is this entire notebook in four words.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Stress the compaction.** Add two more facts in early turns (a favourite colour, a meeting time), let compaction fire, then quiz the chat on all of them. Then sabotage it: change the summarizer's instructions to "Summarize in one short sentence." **Done when:** you can name which facts died, and explain why the summary prompt — not the mechanism — decides what survives.
2. **Break the subagent's isolation on purpose.** Make `task()` return the subagent's *entire* message transcript instead of its answer, re-run the demo, and compare `main_tokens` before and after. **Done when:** you can state the number: how many extra tokens the main conversation swallowed for zero extra information.
3. **Resumable sessions.** Make `write_todos` also save the plan to `todos.md` (offloading!). Then simulate a crash: start a *fresh* `mini_agent` whose system prompt includes the contents of `todos.md`, with the task "continue where the previous session left off". **Done when:** the new session picks up from the first non-DONE step instead of starting over.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Stress the compaction:** with the one-sentence summarizer, typically the *latest* topic survives and the early facts (name, colour, meeting time) vanish — the summariser keeps what looks salient, and brevity forces triage. The fix is never "summarize harder", it's telling the summarizer **what classes of information are non-negotiable** (facts about the user, decisions, open tasks). Compare your broken summary with the original prompt's checklist and the difference is the whole lesson.

**2 — Break the isolation:**

```python
def leaky_task(prompt: str) -> str:
    # returns EVERYTHING the subagent saw — the anti-pattern
    answer, tokens = mini_agent(prompt, doc_tools, doc_tool_functions,
        system="You are a research subagent.", label="    sub ")
    transcript_dump = answer + "\n\n[full log below]\n" + (WORKSPACE / "meeting_log.txt").read_text()
    return transcript_dump
```

Main-conversation tokens typically jump several-fold, because the dump gets re-sent on **every subsequent turn** — you don't pay for a big tool result once, you pay for it for the rest of the session. That re-payment is the deep reason subagents return summaries.

**3 — Resumable sessions:**

```python
def write_todos(todos: list) -> str:
    global TODOS
    TODOS = todos
    (WORKSPACE / "todos.md").write_text("\n".join(todos))     # offload
    print("    [plan] " + " | ".join(todos))
    return "Current plan:\n" + "\n".join(f"- {t}" for t in todos)

saved = (WORKSPACE / "todos.md").read_text()
answer, _ = mini_agent(
    "Continue where the previous session left off.",
    todo_tools, {"write_todos": write_todos, "create_file": create_file},
    system=("You are resuming an interrupted session. The plan so far:\n"
            f"{saved}\n\nSkip steps marked DONE. Update the plan as you go."))
```

**Done when** the first tool call of the new session is not a re-do of step 1. State on disk + a plan format the next session can parse = the minimum viable "durable agent". LangGraph makes this a first-class feature (checkpointers, 2.3); now you know exactly what problem they solve.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| Compaction fires every single turn | `COMPACT_AT` below the natural size of system prompt + a couple of turns | Raise the threshold; real harnesses trigger near the model's limit, not at 700 tokens (ours is low *for the demo*). Claude Code even stops auto-compacting if it detects thrashing |
| Facts vanish after compaction | Summary prompt doesn't demand them | Enumerate the non-negotiables in the summarizer instructions (facts about the user, decisions, task state, next steps) |
| The subagent answers from its own imagination without reading | Its system prompt doesn't name the files, or the tool description undersells reading | Name the available files explicitly in the subagent's system prompt (ours does) |
| Subagent hits max turns on a long file | `limit` too small → too many reads needed | Raise `limit` or `max_turns`; or give it a search tool instead of sequential reads (0.7 exercise 1) |
| The model skips `write_todos` on multi-step tasks | The ritual isn't mandated | The system prompt must *require* it ("first call write_todos"), not suggest it — suggestions lose to task momentum |
| `BadRequestError` about message ordering after compaction | A custom `compact()` left the list starting with an assistant message or split a tool-call pair | Keep the user+assistant bridge pair our `compact()` inserts; never cut between an assistant tool-call message and its tool results |
::::

**Your context toolkit is complete.** You can now compact old history, isolate expensive research in subagents, anchor a long plan, offload state to files, and keep application metadata out of the provider feed. Those mechanisms solve different failure modes; production harnesses compose them rather than searching for one clever memory system.

**What's next — 0.9:** the loop, tools, permissions and context are still separate notebook-sized ideas. We assemble them into a small **agent harness** with provider adapters, domain events, a model-facing feed and invariants you can test — the bridge from a loop to a product.